# AQI Data Warehouse — Exploratory Analysis

Individual analysis notebook, based on the `fact_air_quality` / `dim_city` / `dim_time`
star schema loaded in our Neon Postgres warehouse.

**Goals of this notebook:**
1. Connect to the warehouse and load the data into a single analysis-ready DataFrame
2. Overview of coverage (rows per city, date range, missing values)
3. AQI comparison across the 5 cities
4. Time trends (daily/weekly evolution, pollution peaks)
5. Correlations between pollutants
6. Weekday vs weekend and hour-of-day patterns

In [10]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

DATABASE_URL = os.getenv("DATABASE_URL")

#print(f"✅ DATABASE_URL found: {DATABASE_URL[:30]}..." if DATABASE_URL else "❌ DATABASE_URL not found! Create a .env file with DATABASE_URL=...")
if not DATABASE_URL:
    raise ValueError("❌ Nooooooooooooo DATABASE_URL not found! Create a .env file with DATABASE_URL=...")

engine = create_engine(DATABASE_URL)
print("Yessssss the data_warehouse is connected successfully!" if engine else "Connection object not created.")

Yessssss the data_warehouse is connected successfully!


## 1. Load the data

We join the fact table with both dimensions into one flat DataFrame — the most convenient shape for exploratory analysis in pandas.

In [5]:
query = """
SELECT
    c.city_name,
    c.country,
    c.latitude,
    c.longitude,
    t.timestamp_utc,
    t.date,
    t.hour,
    t.day_of_week,
    t.day_name,
    t.is_weekend,
    f.aqi,
    f.co, f.no, f.no2, f.o3, f.so2, f.pm2_5, f.pm10, f.nh3
FROM fact_air_quality f
JOIN dim_city c ON c.city_key = f.city_key
JOIN dim_time t ON t.time_key = f.time_key
ORDER BY c.city_name, t.timestamp_utc
"""

df = pd.read_sql(query, engine)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
print(f"{len(df):,} rows loaded, {df['city_name'].nunique()} cities")
df

10,368 rows loaded, 5 cities


,city_name,country,latitude,longitude,timestamp_utc,date,hour,day_of_week,day_name,is_weekend,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,Antananarivo,MG,-18.8792,47.5079,2026-04-26 12:00:00+00:00,2026-04-26,12,6,Dimanche,True,1,62.86,0.03,0.10,43.89,0.12,0.87,1.58,0.41
1,Antananarivo,MG,-18.8792,47.5079,2026-04-26 13:00:00+00:00,2026-04-26,13,6,Dimanche,True,1,67.81,0.05,0.19,41.59,0.14,1.07,2.00,0.64
2,Antananarivo,MG,-18.8792,47.5079,2026-04-26 14:00:00+00:00,2026-04-26,14,6,Dimanche,True,1,76.20,0.05,0.41,37.69,0.16,1.36,2.67,0.92
3,Antananarivo,MG,-18.8792,47.5079,2026-04-26 15:00:00+00:00,2026-04-26,15,6,Dimanche,True,1,85.01,0.01,0.73,34.26,0.18,1.67,3.34,1.11
4,Antananarivo,MG,-18.8792,47.5079,2026-04-26 16:00:00+00:00,2026-04-26,16,6,Dimanche,True,1,94.56,0.00,0.96,30.59,0.20,2.05,4.23,1.29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10363,Paris,FR,48.8566,2.3522,2026-07-25 07:00:00+00:00,2026-07-25,7,5,Samedi,True,1,125.21,0.51,4.91,59.76,0.75,5.45,7.40,5.70
10364,Paris,FR,48.8566,2.3522,2026-07-25 08:00:00+00:00,2026-07-25,8,5,Samedi,True,2,125.84,0.94,4.76,60.96,0.82,5.73,7.76,5.64
10365,Paris,FR,48.8566,2.3522,2026-07-25 09:00:00+00:00,2026-07-25,9,5,Samedi,True,2,125.60,1.04,4.33,67.10,0.93,6.13,8.25,5.43
10366,Paris,FR,48.8566,2.3522,2026-07-25 10:00:00+00:00,2026-07-25,10,5,Samedi,True,2,115.95,0.32,1.80,95.92,0.73,5.54,7.19,3.34


## Partie 1 : Nettoyage et etude de notre data warehouse
### 1. Compter le nombre total de lignes

In [6]:
# Méthode 1 : La plus simple
total_rows = len(df)
print(f"📊 Nombre total de mesures: {total_rows:,}")

# Méthode 2 : Shape (donne (lignes, colonnes))
print(f"📊 Dimensions: {df.shape}")

📊 Nombre total de mesures: 10,368
📊 Dimensions: (10368, 19)


### 2. Compter par ville (groupby)

In [8]:
# Nombre de mesures par ville
city_counts = df['city_name'].value_counts()
print("\n🏙️ Nombre de mesures par ville:")
print(city_counts)



🏙️ Nombre de mesures par ville:
city_name
Antananarivo    2088
Nairobi         2088
Paris           2088
Beijing         2064
New Delhi       2040
Name: count, dtype: int64
